# HD-97658b RM Retrieval + Analysis

Set `do_retrieval = True` to run EDMCMC (same workflow as the planet retrieval `.py` script).
Set `do_retrieval = False` to reuse saved chains/best-fit CSV and only regenerate trace, corner, and RV-model plots.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import corner
import sys
from pathlib import Path


def bootstrap_paths_for_modules(start_path: Path) -> None:
    for candidate in [start_path, *start_path.parents]:
        run_candidate = candidate if (candidate / "edmcmc.py").exists() else candidate / "run"
        if (run_candidate / "edmcmc.py").exists():
            repo_candidate = run_candidate.parent
            if str(run_candidate) not in sys.path:
                sys.path.insert(0, str(run_candidate))
            if str(repo_candidate) not in sys.path:
                sys.path.insert(0, str(repo_candidate))
            return


start_path_for_imports = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd().resolve()
bootstrap_paths_for_modules(start_path_for_imports)

import ellc
from ellc import lc
import edmcmc as edm

# Control flags
do_retrieval = False
custom_bestfitting_params = False
custom_vsini = 0.5
custom_lambda1 = 45


In [ ]:
def find_run_dir(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data").is_dir() and (candidate / "edmcmc.py").exists():
            return candidate
        run_candidate = candidate / "run"
        if (run_candidate / "data").is_dir() and (run_candidate / "edmcmc.py").exists():
            return run_candidate
    raise FileNotFoundError("Could not find the run/ directory containing data/ and edmcmc.py")


start_path = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd().resolve()
run_dir = find_run_dir(start_path)

planet_name = "HD-97658b"
model_name = "best"
csvfile = run_dir / "data" / planet_name / "HD97658_2025Feb20.csv"
output_dir = run_dir / "edmcmc_output" / planet_name
output_dir.mkdir(parents=True, exist_ok=True)

chains_file = output_dir / f"{planet_name}_{model_name}_chains.npz"
bestfit_csv = output_dir / f"{planet_name}_{model_name}_bestfit.csv"

print(f"run_dir: {run_dir}")
print(f"csvfile: {csvfile}")
print(f"output_dir: {output_dir}")

df = pd.read_csv(csvfile, comment='#')
for col in ("ccfjdsum", "ccfrvmod", "dvrms"):
    if col not in df.columns:
        raise ValueError(f"Input CSV missing required column: {col}")

time_obs = df['ccfjdsum'].values.astype(float)
rv_data = df['ccfrvmod'].values.astype(float)
rv_err = df['dvrms'].values.astype(float)


In [ ]:
# Fixed system parameters
r_1 = 0.04132231404
r_2 = 0.0011228161490683
incl = 89.45
a = 24.2
e = 0.05
f_c = np.sqrt(e) * math.cos(np.deg2rad(-10))
f_s = np.sqrt(e) * math.sin(np.deg2rad(-10))
q = 0.0000291991764706
shape_1 = "sphere"
shape_2 = "sphere"
sbratio = 0.0
period = 9.489
t0 = 2460726.88970225

# Compute weighted systemic offset using out-of-transit points and subtract it
i_rad = math.radians(incl)
rsum = (r_1 + r_2)
val = rsum / max(1e-12, math.sin(i_rad))
if val >= 1.0:
    transit_duration_days = 0.2
else:
    transit_duration_days = period / math.pi * val

transit_half_phase = (transit_duration_days / 2.0) / period
phases_for_mask = ((time_obs - t0) / period + 0.5) % 1.0 - 0.5
in_transit_mask = np.abs(phases_for_mask) < transit_half_phase
out_of_transit_mask = ~in_transit_mask

if out_of_transit_mask.sum() < 3:
    out_of_transit_mask = np.ones_like(out_of_transit_mask, dtype=bool)
weights = 1.0 / (rv_err**2)
gamma_weighted = np.sum(weights[out_of_transit_mask] * rv_data[out_of_transit_mask]) / np.sum(weights[out_of_transit_mask])

rv_data = rv_data - gamma_weighted


In [ ]:
def loglikelihood(p, time, rv_obs, rv_err):
    vsini_mod, lambda1_mod = p

    rv_model, _ = ellc.rv(
        time,
        t_zero=t0,
        period=period,
        lambda_1=lambda1_mod,
        radius_1=r_1,
        radius_2=r_2,
        incl=incl,
        a=a,
        f_c=f_c,
        f_s=f_s,
        q=q,
        shape_1=shape_1,
        shape_2=shape_2,
        vsini_1=vsini_mod,
        flux_weighted=True,
        sbratio=sbratio,
        verbose=0,
    )

    chi2 = np.sum((rv_obs - rv_model) ** 2 / (rv_err) ** 2)
    return -0.5 * chi2


labels = ['v_sini', 'lambda']
vsini_guess = 0.5
lambda1_guess = 45
p0 = [vsini_guess, lambda1_guess]
wid = [0.0001, 0.0001]
parinfo = [
    {
        'fixed': False,
        'limits': [max(0.0, vsini_guess - 0.5), vsini_guess + 1.5],
        'limited': [True, True],
    },
    {
        'fixed': False,
        'limits': [lambda1_guess - 25.0, lambda1_guess + 25.0],
        'limited': [True, True],
    },
]
ndim = len(p0)


In [ ]:
if do_retrieval:
    out = edm.edmcmc(
        loglikelihood,
        p0,
        wid,
        args=(time_obs, rv_data, rv_err),
        parinfo=parinfo,
        nwalkers=10,
        nlink=10000,
        nburnin=500,
        ncores=8,
        quiet=True,
    )

    np.savez(chains_file, flatchains=out.flatchains, whichlink=out.whichlink)

    bestfits = {
        'parameter': ['vsin_i', 'lambda'],
        'median': [np.median(out.flatchains[:, 0]), np.median(out.flatchains[:, 1])],
        'std': [np.std(out.flatchains[:, 0]), np.std(out.flatchains[:, 1])],
    }
    pd.DataFrame(bestfits).to_csv(bestfit_csv, index=False)

    flatchains = out.flatchains
    whichlink = out.whichlink
    best_vsini = float(np.median(flatchains[:, 0]))
    best_lambda = float(np.median(flatchains[:, 1]))

    print(f"Best-fitting parameters saved to: {bestfit_csv}")
    print(f"Chains saved to: {chains_file}")
else:
    if not chains_file.exists():
        raise FileNotFoundError(
            f"Missing chains file: {chains_file}. Run once with do_retrieval=True to create it."
        )
    if not bestfit_csv.exists():
        raise FileNotFoundError(
            f"Missing best-fit file: {bestfit_csv}. Run once with do_retrieval=True to create it."
        )

    chain_data = np.load(chains_file)
    flatchains = chain_data['flatchains']
    whichlink = chain_data['whichlink']

    best_df = pd.read_csv(bestfit_csv)
    best_vsini_csv = float(best_df.loc[best_df['parameter'] == 'vsin_i', 'median'].iloc[0])
    best_lambda_csv = float(best_df.loc[best_df['parameter'] == 'lambda', 'median'].iloc[0])

    if custom_bestfitting_params:
        best_vsini = float(custom_vsini)
        best_lambda = float(custom_lambda1)
    else:
        best_vsini = best_vsini_csv
        best_lambda = best_lambda_csv

print(np.median(flatchains[:,0]), '+/-', np.std(flatchains[:,0]), ';    ', np.median(flatchains[:,1]), '+/-', np.std(flatchains[:,1]))
print(f"Using best-fit values for RV model: vsini={best_vsini}, lambda={best_lambda}")


In [ ]:
# Trace plot
fig1, axes1 = plt.subplots(ndim, figsize=(10, 1 + 2 * ndim), sharex=True)
for i in range(ndim):
    ax = axes1[i]
    ax.plot(whichlink, flatchains[:, i], '.')
    ax.set_ylabel(labels[i])
axes1[-1].set_xlabel("Link number")
fig1_name = output_dir / f"{planet_name}_{model_name}_trace.pdf"
fig1.savefig(fig1_name)
print('walker trace plot:' + str(fig1_name))
plt.close(fig1)

# Corner plot
fig2 = plt.figure(figsize=(1 + 3 * ndim, 1 + 3 * ndim))
fig2 = corner.corner(flatchains, labels=labels)
fig2_name = output_dir / f"{planet_name}_{model_name}_corner.pdf"
fig2.savefig(fig2_name)
print('corner plot:' + str(fig2_name))
plt.close(fig2)


In [ ]:
# RV model plot (with posterior bands from samples)
all_samples = flatchains
nsamples_total = all_samples.shape[0]
nsamp = min(1000, nsamples_total)
rng = np.random.default_rng(12345)
sel_idx = rng.choice(nsamples_total, size=nsamp, replace=False)

ntime = len(time_obs)
models = np.zeros((nsamp, ntime))
for j, idx in enumerate(sel_idx):
    samp_vsini = all_samples[idx, 0]
    samp_lambda = all_samples[idx, 1]

    rv_call_samp = ellc.rv(
        time_obs,
        t_zero=t0,
        period=period,
        lambda_1=samp_lambda,
        radius_1=r_1,
        radius_2=r_2,
        incl=incl,
        a=a,
        f_c=f_c,
        f_s=f_s,
        q=q,
        shape_1=shape_1,
        shape_2=shape_2,
        vsini_1=samp_vsini,
        flux_weighted=True,
        sbratio=sbratio,
        verbose=0,
    )

    star_rv_samp = np.asarray(rv_call_samp[0])
    sys_offset_samp = np.median(rv_data) - np.median(star_rv_samp)
    star_rv_samp += sys_offset_samp
    models[j, :] = star_rv_samp

median_model = np.median(models, axis=0)
p16 = np.percentile(models, 16.0, axis=0)
p84 = np.percentile(models, 84.0, axis=0)
p025 = np.percentile(models, 2.5, axis=0)
p975 = np.percentile(models, 97.5, axis=0)

rv_call_best = ellc.rv(
    time_obs,
    t_zero=t0,
    period=period,
    lambda_1=best_lambda,
    radius_1=r_1,
    radius_2=r_2,
    incl=incl,
    a=a,
    f_c=f_c,
    f_s=f_s,
    q=q,
    shape_1=shape_1,
    shape_2=shape_2,
    vsini_1=best_vsini,
    flux_weighted=True,
    sbratio=sbratio,
    verbose=0,
)
star_rv_best = np.asarray(rv_call_best[0])

sys_offset_best = np.median(rv_data) - np.median(star_rv_best)
star_rv_best += sys_offset_best

fig3, (ax_top, ax_bot) = plt.subplots(
    2, 1, sharex=True, figsize=(10, 8), gridspec_kw={'height_ratios': [3, 1]}
)

font_choice = 'serif'
label_fontsize = 14
tick_fontsize = 12
legend_fontsize = 12
marker_size = 5
model_linewidth = 1.5
band_alpha_1sig = 0.34
band_alpha_2sig = 0.16

ax_top.fill_between(time_obs, p025 * 1e3, p975 * 1e3, color='red', alpha=band_alpha_2sig, linewidth=0.0)
ax_top.fill_between(time_obs, p16 * 1e3, p84 * 1e3, color='red', alpha=band_alpha_1sig, linewidth=0.0)

ax_top.errorbar(
    time_obs,
    rv_data * 1e3,
    yerr=rv_err * 1e3,
    fmt='o',
    ms=marker_size,
    c='k',
    label='Data',
    zorder=5,
)
ax_top.plot(
    time_obs,
    star_rv_best * 1e3,
    '-',
    lw=model_linewidth,
    alpha=1.0,
    c='red',
    label='Median Model',
)
ax_top.tick_params(axis='both', labelsize=tick_fontsize)
ax_top.legend(prop={'size': legend_fontsize, 'family': font_choice}, loc='best')

residuals_ms = (rv_data - median_model) * 1e3
ax_bot.errorbar(
    time_obs,
    residuals_ms,
    yerr=rv_err * 1e3,
    fmt='o',
    ms=marker_size,
    c='k',
    label='residuals',
)
ax_bot.axhline(0.0, color='red', linestyle='-', alpha=0.7)
ax_bot.set_xlabel('Time (BJD)', fontsize=label_fontsize, fontname=font_choice)
ax_bot.tick_params(axis='both', labelsize=tick_fontsize)

for ax in (ax_top, ax_bot):
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontname(font_choice)

fig3.text(0.02, 0.5, 'Radial velocity (m/s)', va='center', rotation='vertical',
          fontsize=label_fontsize, fontname=font_choice)

plt.tight_layout(rect=[0.03, 0.03, 1, 0.98])

if (not do_retrieval) and custom_bestfitting_params:
    rv_model_tag = f"custom_vsin{best_vsini:.6g}_lambda{best_lambda:.6g}"
    fig3_name = output_dir / f"{planet_name}_{model_name}_{rv_model_tag}_rv_model.pdf"
else:
    fig3_name = output_dir / f"{planet_name}_{model_name}_rv_model.pdf"

fig3.savefig(fig3_name)
print('bestmodel plot:' + str(fig3_name))
plt.show()
plt.close(fig3)
